# CubicasA5K — YOLOv8 Floor Plan Detection (Kaggle P100)
**Classes:** door · wall · window  
**Dataset:** 4178 train / 400 val / 400 test  

### Before running:
1. Add dataset via **+ Add Input** (upload cubicasa5k.zip)
2. Settings → Accelerator → **GPU P100**
3. Settings → Internet → **ON**
4. Run All


In [ ]:
# ── Cell 1: Ensure PyTorch is available (Kaggle/P100-safe) ───────────────────
import os
import sys
import subprocess

marker = '/tmp/torch_fixed'

# 1) Prefer already-installed torch (Kaggle usually has a working GPU build preloaded)
try:
    import torch
    print(f'Existing PyTorch: {torch.__version__}')
    print(f'CUDA available : {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU            : {torch.cuda.get_device_name(0)}')
        print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
except Exception:
    torch = None


def pip_install(pkgs, extra_args=None):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--timeout', '120', '--retries', '10']
    if extra_args:
        cmd.extend(extra_args)
    cmd.extend(pkgs)
    subprocess.run(cmd, check=True)


need_install = True
if 'torch' in globals() and torch is not None and torch.cuda.is_available():
    need_install = False

if need_install and not os.path.exists(marker):
    print('Installing compatible PyTorch build...')

    # Try plain PyPI versions first (your runtime shows these are available)
    candidates = [
        (['torch==2.2.2', 'torchvision==0.17.2'], None),
        (['torch==2.3.1', 'torchvision==0.18.1'], None),
        (['torch==2.2.2+cu118', 'torchvision==0.17.2+cu118'], ['--index-url', 'https://download.pytorch.org/whl/cu118']),
    ]

    installed = False
    last_err = None
    for pkgs, extra in candidates:
        try:
            print(f'Trying: {pkgs[0]} / {pkgs[1]}')
            pip_install(pkgs, extra)
            installed = True
            break
        except subprocess.CalledProcessError as e:
            last_err = e
            print('Failed, trying next candidate...')

    if not installed:
        raise RuntimeError('Could not install a compatible torch build') from last_err

    open(marker, 'w').close()
    print('Done. Restarting kernel...')
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
elif need_install:
    print('Install marker exists but torch CUDA is still unavailable. Remove /tmp/torch_fixed and rerun cell.')

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Install ultralytics ───────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
import ultralytics
print(f'Ultralytics: {ultralytics.__version__}')


In [ ]:
# ── Cell 3: Locate dataset ────────────────────────────────────────────────────
import os, glob

# Check if cubicasa5k-2-6 folder is already present (uploaded as dataset)
existing = glob.glob('/kaggle/input/**/cubicasa5k-2-6', recursive=True)

if existing:
    DATASET_ROOT = existing[0]
    print(f'Dataset found at: {DATASET_ROOT}')
else:
    # Try unzipping from zip
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    if zips:
        EXTRACT_TO = '/kaggle/working/datasets'
        os.makedirs(EXTRACT_TO, exist_ok=True)
        print(f'Unzipping {zips[0]} ...')
        os.system(f'unzip -q -o "{zips[0]}" -d "{EXTRACT_TO}"')
        DATASET_ROOT = f'{EXTRACT_TO}/cubicasa5k-2-6'
        print('Done.')
    else:
        raise FileNotFoundError('No dataset found! Add cubicasa5k via + Add Input.')

print(f'Train images: {len(os.listdir(os.path.join(DATASET_ROOT, "train", "images")))}')
print(f'Val images  : {len(os.listdir(os.path.join(DATASET_ROOT, "valid", "images")))}')


In [ ]:
# ── Cell 4: Write data.yaml ───────────────────────────────────────────────────
import yaml

DATA_YAML_PATH = '/kaggle/working/data.yaml'

data = {
    'path':  DATASET_ROOT,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc': 3,
    'names': ['door', 'wall', 'window']
}

with open(DATA_YAML_PATH, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

os.system(f'cat {DATA_YAML_PATH}')
print(f'data.yaml written to {DATA_YAML_PATH}')


In [ ]:
# ── Cell 5: Model settings for P100 (16GB VRAM, sm_60) ───────────────────────
MODEL = 'yolov8m.pt'   # medium model — P100 16GB can handle it
BATCH = 8              # conservative for dense floor plan boxes
IMGSZ = 800            # good balance of detail vs memory

print(f'Model: {MODEL}  |  Batch: {BATCH}  |  Imgsz: {IMGSZ}')


In [ ]:
# ── Cell 6: TRAIN ─────────────────────────────────────────────────────────────
from ultralytics import YOLO

model = YOLO(MODEL)

results = model.train(
    data             = DATA_YAML_PATH,
    epochs           = 80,
    imgsz            = IMGSZ,
    batch            = BATCH,
    device           = 0,
    workers          = 4,
    cache            = 'disk',
    patience         = 20,
    multi_scale      = True,
    cos_lr           = True,
    close_mosaic     = 15,
    optimizer        = 'AdamW',
    lr0              = 0.001,
    lrf              = 0.01,
    warmup_epochs    = 3,
    warmup_momentum  = 0.8,
    weight_decay     = 0.0005,
    box              = 7.5,
    cls              = 0.5,
    dfl              = 1.5,
    iou              = 0.7,
    mosaic           = 1.0,
    mixup            = 0.1,
    copy_paste       = 0.15,
    degrees          = 10.0,
    translate        = 0.1,
    scale            = 0.6,
    shear            = 2.0,
    fliplr           = 0.5,
    flipud           = 0.25,
    hsv_h            = 0.015,
    hsv_s            = 0.7,
    hsv_v            = 0.4,
    perspective      = 0.0001,
    project          = '/kaggle/working/runs',
    name             = 'cubicasa_yolo',
    exist_ok         = True,
)


In [ ]:
# ── Cell 7: Validate best model ───────────────────────────────────────────────
from ultralytics import YOLO

best_model = YOLO('/kaggle/working/runs/cubicasa_yolo/weights/best.pt')

metrics = best_model.val(
    data   = DATA_YAML_PATH,
    imgsz  = IMGSZ,
    split  = 'val',
    iou    = 0.45,
    conf   = 0.001,
    device = 0,
)

print('\n========== RESULTS ==========')
print(f'Precision : {metrics.box.mp:.4f}')
print(f'Recall    : {metrics.box.mr:.4f}')
print(f'mAP@50    : {metrics.box.map50:.4f}')
print(f'mAP@50-95 : {metrics.box.map:.4f}')
print('==============================')
for i, name in enumerate(['door', 'wall', 'window']):
    p  = metrics.box.p[i]
    r  = metrics.box.r[i]
    m  = metrics.box.ap50[i]
    f1 = 2 * p * r / (p + r + 1e-9)
    print(f'  {name:6s}  P={p:.3f}  R={r:.3f}  mAP50={m:.3f}  F1={f1:.3f}')


In [ ]:
# ── Cell 8: Show training curves ──────────────────────────────────────────────
from IPython.display import Image, display
import os

run_dir = '/kaggle/working/runs/cubicasa_yolo'
for img in ['results.png', 'confusion_matrix.png', 'BoxPR_curve.png']:
    path = os.path.join(run_dir, img)
    if os.path.exists(path):
        print(f'--- {img} ---')
        display(Image(path))


In [ ]:
# ── Cell 9: Output ────────────────────────────────────────────────────────────
# best.pt is automatically saved to Kaggle Output tab
# Download it from: Notebook → Output → runs/cubicasa_yolo/weights/best.pt
import os
best = '/kaggle/working/runs/cubicasa_yolo/weights/best.pt'
size = os.path.getsize(best) / 1e6
print(f'best.pt saved: {size:.1f} MB')
print('Download from Kaggle Output tab on the right panel.')
